# ChurnGuard research prototype
This notebook is intentionally preserved as exploratory research code for the Assignment II research-vs-production comparison.

In [ ]:
import pandas as pd, numpy as np, matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, accuracy_score
%matplotlib inline

In [ ]:
df = pd.read_csv("../data/raw/telco_churn.csv")  # hard-coded relative path
df.head()

In [ ]:
df.shape, df.churn.mean()

In [ ]:
df.isnull().sum()

In [ ]:
# just fill the nulls, figure out something better later
df = df.fillna(0)

In [ ]:
# one hot everything
X = pd.get_dummies(df.drop(['churn', 'customer_id'], axis=1))
y = df.churn

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)  # no random_state

In [ ]:
m = RandomForestClassifier(n_estimators=100)
m.fit(X_train, y_train)
p = m.predict_proba(X_test)[:, 1]
print(accuracy_score(y_test, m.predict(X_test)))
print(roc_auc_score(y_test, p))

In [ ]:
# try adding a feature
X_train['spm'] = X_train.total_charges / X_train.tenure_months
X_test['spm'] = X_test.total_charges / X_test.tenure_months
m.fit(X_train, y_train)
roc_auc_score(y_test, m.predict_proba(X_test)[:, 1])

In [ ]:
# hmm worse. try again with different depth
m2 = RandomForestClassifier(n_estimators=100, max_depth=8)
m2.fit(X_train, y_train)
roc_auc_score(y_test, m2.predict_proba(X_test)[:, 1])

In [ ]:
import pickle
pickle.dump(m2, open('model.pkl', 'wb'))  # bare estimator, no transforms